In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report

In [ ]:
#Reads the text file, extracts the headline text + sentiment labels,
#removes malformed/empty rows, standardizes labels, and stores everything
#in a pandas dataframe

FILE_PATH = "/content/Sentences_75Agree.txt"

rows = []

with open(FILE_PATH, "r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

for line in lines:
    line = line.strip()
    if not line:
        continue

    parts = line.rsplit("@", 1)

    if len(parts) != 2:
        continue

    text, label = parts
    text = text.strip()
    label = label.strip().lower()

    if text and label in ["positive", "neutral", "negative"]:
        rows.append((text, label))

df = pd.DataFrame(rows, columns=["text", "label"])

print(df.shape)
print(df.head())
print(df["label"].value_counts())

df.columns = ["text", "label"]

In [ ]:
#Showing the distribution of the different classes
#(positive, negative, neutral)
print(df["label"].value_counts())

df["label"].value_counts().plot(kind="bar")

plt.title("Class Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")

plt.show()

In [ ]:
#Shows average sentence length for each sentiment class
df["length"] = df["text"].apply(lambda x: len(x.split()))

print(df.groupby("label")["length"].mean())

In [ ]:
#Splits dataset into training and test sets, converts the text into
#TF-IDF feature vectors (unigrams + bigrams), trains a new Logistic Regression
#classifier, and evaluates model performance.

X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1,2),
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train_tfidf, y_train)

preds = model.predict(X_test_tfidf)

print(classification_report(y_test, preds))

In [ ]:
#Analyzes which words most strongly influence each sentiment class in the LR
#model by looking at the TF-IDF feature weights. It also identifies the most common bigrams

feature_names = vectorizer.get_feature_names_out()

coef = model.coef_

class_names = model.classes_

TOP_N = 15

for i, class_name in enumerate(class_names):

    top_idx = np.argsort(coef[i])[-TOP_N:]

    top_words = feature_names[top_idx]
    top_weights = coef[i][top_idx]

    df_words = pd.DataFrame({
        "word": top_words,
        "weight": top_weights
    }).sort_values(by="weight", ascending=False)

    print(f"\nTop words for class: {class_name}")
    print(df_words)

    from sklearn.feature_extraction.text import CountVectorizer

bigram_vectorizer = CountVectorizer(
    ngram_range=(2,2),
    stop_words='english'
)

X_bigrams = bigram_vectorizer.fit_transform(df["text"])

sum_words = X_bigrams.sum(axis=0)

words_freq = [
    (word, sum_words[0, idx])
    for word, idx in bigram_vectorizer.vocabulary_.items()
]

words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

print("\nTop Bigrams:\n")

for word, freq in words_freq[:20]:
    print(word, int(freq))

In [ ]:
#Applies K-means clustering to the TF-IDF feature vectors to see
#themes/groupings in the dataset. The most important terms in each cluster
#are extracted.

X_full = vectorizer.fit_transform(df["text"])

NUM_CLUSTERS = 5

kmeans = KMeans(
    n_clusters=NUM_CLUSTERS,
    random_state=42
)

kmeans.fit(X_full)

df["cluster"] = kmeans.labels_

terms = vectorizer.get_feature_names_out()

for i in range(NUM_CLUSTERS):

    center = kmeans.cluster_centers_[i]

    top_idx = np.argsort(center)[-10:]

    top_terms = [terms[j] for j in top_idx]

    print(f"\nCluster {i}")
    print(top_terms)